# 🥇 Gold — 5 tabelas analíticas

Dados agregados, prontos para consumo direto por API, dashboard e agente de IA.

|  |  |
|---|---|
| **Lê** | `silver.flights_clean` + `silver.dim_airports` |
| **Grava** | `gold.airline_performance`, `gold.airport_performance`, `gold.route_performance`, `gold.delay_causes`, `gold.flight_trends` |
| **Regido por** | `business_rules.md` (KPIs 01–09) · `business_questions.md` (as 16 perguntas) |

**Regra da camada:** só existe tabela aqui que responda a uma pergunta de negócio real e implemente um KPI documentado. Nenhuma foi criada "porque estava no plano".

| Tabela | Granularidade | Linhas |
|---|---|---:|
| `airline_performance` | 1 por companhia | 15 |
| `airport_performance` | 1 por aeroporto de origem | 348 |
| `route_performance` | 1 por rota (origem + destino) | 6.805 |
| `delay_causes` | agregado único | 1 |
| `flight_trends` | 1 por mês | 12 |

### Decisões desta camada

- **Limiar de atraso = 15 min** (`LIMIAR_ATRASO`) — padrão oficial do BTS, não foi inventado para o projeto.
- **O nome do aeroporto entra por `join`, nunca por `groupBy`.** Agrupar pelo nome fundiria ORD com MDW e IAH com HOU, inflando as métricas. A sigla agrega; o nome é colado depois.
- **`broadcast` na dimensão** (~350 linhas) evita o shuffle dos 7M de voos no join.
- **As células após a primeira são inspeção visual** das tabelas criadas (`display`) — não fazem parte do pipeline.

> **Execução com o dataset completo (Etapa 15):** 20,7s. Pipeline inteiro (Bronze → Silver → Gold): ~58s.


In [ ]:
# ============================================================
# ETAPA 15 — GOLD (DATASET COMPLETO)
# Notebook independente — lê silver.flights_clean + silver.dim_airports
# ------------------------------------------------------------
# MUDANÇA DESTA VERSÃO: as tabelas de aeroporto e de rota passam a
# carregar o NOME do aeroporto, não só a sigla IATA.
# A sigla continua sendo a CHAVE (é ela que é única); o nome entra
# como coluna descritiva, vindo de silver.dim_airports.
# ============================================================

import time
from pyspark.sql import functions as F

inicio_execucao = time.time()

df = spark.table("silver.flights_clean")
contagem_silver = df.count()
print("Registros na Silver:", contagem_silver)

# Dimensão de aeroportos (~350 linhas): broadcast evita shuffle dos 7M de voos
df_dim = spark.table("silver.dim_airports")
dim_broadcast = F.broadcast(df_dim)
print("Aeroportos na dimensão:", df_dim.count())

LIMIAR_ATRASO = 15

# 14.1 — gold.airline_performance (sem alteração)
df_airline = (
    df.groupBy("op_unique_carrier")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("arr_delay") > LIMIAR_ATRASO, 1).otherwise(0)).alias("delayed_flights"),
        F.avg("dep_delay").alias("average_departure_delay"),
        F.avg("arr_delay").alias("average_arrival_delay"),
        F.sum("cancelled").alias("cancelled_flights"),
        F.sum("diverted").alias("diverted_flights"),
    )
    .withColumn("delay_rate", F.round(F.col("delayed_flights") / F.col("total_flights"), 4))
    .withColumn("cancellation_rate", F.round(F.col("cancelled_flights") / F.col("total_flights"), 4))
    .orderBy(F.col("total_flights").desc())
)
df_airline.write.format("delta").mode("overwrite").saveAsTable("gold.airline_performance")
print("gold.airline_performance:", df_airline.count(), "companhias")

# 14.2 — gold.airport_performance (agora com nome do aeroporto)
# A agregação continua por 'origin' (a sigla): é ela que identifica o
# aeroporto. O nome é adicionado DEPOIS, por join — agrupar pelo nome
# fundiria ORD com MDW, IAH com HOU, DCA com IAD.
df_airport = (
    df.groupBy("origin")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("arr_delay") > LIMIAR_ATRASO, 1).otherwise(0)).alias("delayed_flights"),
        F.avg("dep_delay").alias("average_departure_delay"),
        F.sum("cancelled").alias("cancelled_flights"),
    )
    .withColumnRenamed("origin", "airport")
    .join(dim_broadcast, F.col("airport") == F.col("airport_code"), "left")
    .withColumn("delay_rate", F.round(F.col("delayed_flights") / F.col("total_flights"), 4))
    .withColumn("cancellation_rate", F.round(F.col("cancelled_flights") / F.col("total_flights"), 4))
    .select(
        "airport",
        "airport_name",
        "airport_city",
        "airport_state",
        "airport_label",
        "total_flights",
        "delayed_flights",
        "average_departure_delay",
        "cancelled_flights",
        "delay_rate",
        "cancellation_rate",
    )
    .orderBy(F.col("total_flights").desc())
)
df_airport.write.format("delta").mode("overwrite").saveAsTable("gold.airport_performance")
print("gold.airport_performance:", df_airport.count(), "aeroportos")

# 14.3 — gold.route_performance (nome da origem e do destino)
# Dois joins com a mesma dimensão: um alias para cada ponta da rota.
dim_origem = dim_broadcast.select(
    F.col("airport_code").alias("cod_origem"),
    F.col("airport_name").alias("origin_name"),
)
dim_destino = dim_broadcast.select(
    F.col("airport_code").alias("cod_destino"),
    F.col("airport_name").alias("dest_name"),
)

df_route = (
    df.groupBy("origin", "dest")
    .agg(
        F.count("*").alias("total_flights"),
        F.avg("arr_delay").alias("average_arrival_delay"),
        F.avg("distance").alias("average_distance"),
    )
    .join(dim_origem, F.col("origin") == F.col("cod_origem"), "left")
    .join(dim_destino, F.col("dest") == F.col("cod_destino"), "left")
    .select(
        "origin",
        "origin_name",
        "dest",
        "dest_name",
        "total_flights",
        "average_arrival_delay",
        "average_distance",
    )
    .orderBy(F.col("total_flights").desc())
)
df_route.write.format("delta").mode("overwrite").saveAsTable("gold.route_performance")
print("gold.route_performance:", df_route.count(), "rotas")

# 14.4 — gold.delay_causes (sem alteração)
df_delay_causes = df.agg(
    F.sum("carrier_delay").alias("total_carrier_delay"),
    F.sum("weather_delay").alias("total_weather_delay"),
    F.sum("nas_delay").alias("total_nas_delay"),
    F.sum("security_delay").alias("total_security_delay"),
    F.sum("late_aircraft_delay").alias("total_late_aircraft_delay"),
)
df_delay_causes.write.format("delta").mode("overwrite").saveAsTable("gold.delay_causes")
print("gold.delay_causes: 1 registro agregado")

# 14.5 — gold.flight_trends (sem alteração)
df_trends = (
    df.groupBy("month")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("arr_delay") > LIMIAR_ATRASO, 1).otherwise(0)).alias("delayed_flights"),
        F.avg("arr_delay").alias("average_arrival_delay"),
    )
    .withColumn("delay_rate", F.round(F.col("delayed_flights") / F.col("total_flights"), 4))
    .orderBy("month")
)
df_trends.write.format("delta").mode("overwrite").saveAsTable("gold.flight_trends")
print("gold.flight_trends:", df_trends.count(), "meses")

# ------------------------------------------------------------
# QA do enriquecimento: nenhum aeroporto pode ficar sem nome
# ------------------------------------------------------------
sem_nome_aeroporto = df_airport.filter(F.col("airport_name").isNull()).count()
sem_nome_rota = df_route.filter(
    F.col("origin_name").isNull() | F.col("dest_name").isNull()
).count()

print("\n--- QA do join com a dimensão ---")
print("Aeroportos sem nome:", sem_nome_aeroporto)
print("Rotas com alguma ponta sem nome:", sem_nome_rota)

# ============================================================
# RESUMO FINAL (documentar na Etapa 15)
# ============================================================
tempo_total = round(time.time() - inicio_execucao, 1)

print("\n=== RESUMO FINAL — GOLD (DATASET COMPLETO) ===")
print("Registros Silver (entrada):", contagem_silver)
print("Tabelas Gold criadas: 5")
print("Enriquecimento: nome do aeroporto em airport_performance e route_performance")
print(f"Tempo de execução: {tempo_total}s (~{round(tempo_total/60, 1)} min)")
print("Ambiente: Databricks Free Edition — Compute Serverless")

In [ ]:
display(spark.table("silver.dim_airports"))

In [ ]:
display(spark.table("gold.airline_performance"))

In [ ]:
display(spark.table("gold.airport_performance"))

In [ ]:
display(spark.table("gold.route_performance"))

In [ ]:
display(spark.table("gold.delay_causes"))

In [ ]:
display(spark.table("gold.flight_trends"))